# Multi-label Palm Oil Grievance Classification: Fine-tuning TwHIN-BERT

## Overview

This notebook fine-tunes **TwHIN-BERT** for **multi-label** topic classification of palm-oil grievance texts. Each grievance can belong to one *or more* of 6 topics (in contrast to the single-label version). We train on `dominant_topic_results.csv` (80/20 train/val split) and evaluate on a separate held-out test set from `classified_grievances_multilabel.csv`.

| Label | Topic |
|-------|-------|
| 0 | Topic_0 |
| 1 | Topic_1 |
| 2 | Topic_2 |
| 3 | Topic_3 |
| 4 | Topic_4 |
| 5 | Topic_5 |

## Pipeline

1. Install / import dependencies.
2. Load train and test CSVs.
3. **Duplicate & leakage audit** — check for duplicated texts within each split and any cross-set leakage between train/val/test.
4. 80/20 train/val split. **Multi-label stratified** by default (via `iterative-stratification`) so every topic is proportionally represented in both sides.
5. Tokenise with the TwHIN-BERT tokenizer.
6. **Grid search over learning rate** (4 values, everything else fixed at BERT-paper defaults) — simple, explicit, easy to interpret.
7. **K-fold cross-validation on the training set** to pick the sigmoid threshold. Cross-validation gives an unbiased threshold estimate; picking it directly on the validation set would leak.
8. Retrain the final model on the full training set with the best LR, and save the best checkpoint.
9. Evaluate on the held-out test set with the chosen threshold; per-topic accuracy / F1 / macro / micro.
10. Scatter-plot the per-example probabilities with TP / TN / FP / FN colour coding.

## What was fixed in this rewrite

* **Checkpoint save vs load path mismatch** — the previous cells saved with an underscore and loaded with a space, so reloading always failed on a fresh run.
* **`select(range(309))`** hardcoded the train size to 309 rows. Removed — we now use the full split.
* **`val_loss += loss_fct(...)`** was accumulating tensors and holding the autograd graph across epochs. Switched to `.item()`.
* **`hidden_dropout_prob=0.4`** (4x BERT's default) reset to `0.1`.
* **Three inconsistent random seeds** (`random_seed=10`, `set_seed(42)`, `shuffle(seed=1111)`) unified under a single `SEED` constant.
* **Missing gradient clipping** added (`max_norm=1.0`).
* **`num_epochs=50` with a scheduler calibrated to 50** replaced with `num_epochs=15` + early stopping (patience 3). The scheduler now decays over an achievable horizon.
* **`evaluation_threshold=0.1`** — the previous value was extremely low. Replaced with a threshold *chosen from cross-validated OOF predictions*.


# 1. Installations and Imports

In [ ]:
!pip install -q transformers
!pip install -q -U datasets
!pip install -q iterative-stratification

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
import os
import json
from collections import Counter

import numpy as np
import pandas as pd
import torch
from matplotlib import pyplot as plt
from matplotlib.lines import Line2D

from torch.optim import AdamW
from torch.utils.data import DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
    set_seed,
)
from datasets import Dataset

from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import accuracy_score, f1_score, classification_report

from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit, MultilabelStratifiedKFold

from tqdm.notebook import tqdm

# Reduce GPU memory fragmentation on Colab
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Single seed used for split, model init, dataloader shuffle, CV folds
SEED = 42
set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# 2. Configuration

In [ ]:
# --- Model / experiment naming ---------------------------------------------
MODEL_CHECKPOINT = "Twitter/twhin-bert-base"
RUN_NAME         = "20260709_Twitter_multilabel"
NUM_LABELS       = 6
MAX_LENGTH       = 512

# --- Google Drive paths ----------------------------------------------------
DRIVE_ROOT     = "/content/gdrive/MyDrive/Group 3: palm oil topic classifier"
TRAIN_CSV      = f"{DRIVE_ROOT}/Data/Labeled Data/dominant_topic_results.csv"
TEST_CSV       = f"{DRIVE_ROOT}/Text Classification Models/classified_grievances_multilabel.csv"
MODELS_DIR     = f"{DRIVE_ROOT}/Text Classification Models/Twitter_Saved_Models"
BEST_MODEL_PATH = f"{MODELS_DIR}/{RUN_NAME}_best_model.pt"

TOPIC_COLS = [f"Topic_{i}" for i in range(NUM_LABELS)]

# Toggle multi-label stratified splitting; falls back to plain random split if False.
USE_STRATIFIED_SPLIT = True

os.makedirs(MODELS_DIR, exist_ok=True)

# 3. Load Data

`dominant_topic_df` is used for training/validation. `test_df` is a separate held-out corpus. Text columns are named `Text` in the training file and `summary` in the test file — we standardise both to `Text` after loading.

In [ ]:
dominant_topic_df = pd.read_csv(TRAIN_CSV)
test_df           = pd.read_csv(TEST_CSV)

# Standardise column name so downstream code doesn't care which source it came from.
test_df = test_df.rename(columns={"summary": "Text"})[["pk", "Text"] + TOPIC_COLS]

print(f"Train file : {len(dominant_topic_df)} rows | cols: {dominant_topic_df.columns.tolist()}")
print(f"Test file  : {len(test_df)} rows | cols: {test_df.columns.tolist()}")

print("\nPer-topic positive counts in training file:")
print(dominant_topic_df[TOPIC_COLS].sum().astype(int))
print("\nPer-topic positive counts in test file:")
print(test_df[TOPIC_COLS].sum().astype(int))

# Multi-label cardinality (how many topics per document, on average)
print(f"\nAvg labels per doc -- train: {dominant_topic_df[TOPIC_COLS].sum(axis=1).mean():.2f}, "
      f"test: {test_df[TOPIC_COLS].sum(axis=1).mean():.2f}")

# 4. Duplicate & Leakage Audit

We check three things:

1. **Within-file duplicates** — same `Text` appearing more than once inside the training file or the test file. Duplicates inside training just double-count some examples; duplicates that also cross the train/val split boundary silently *inflate* validation metrics.
2. **Train / test leakage** — the same document appearing in both the training source and the test source. If this happens, your reported test metric is optimistically biased because the model has already seen those inputs.
3. **Same text with different labels** — a data-quality signal, not always a bug (multi-label allows the same text to have different multi-hot vectors from different annotators), but worth eyeballing.

We de-duplicate the training file (keeping the first occurrence) *before* the train/val split so no split-boundary leakage is possible. Test duplicates are just reported — we don't drop them from the test corpus so the reported metric matches what the pipeline would see in production.

In [ ]:
def _dup_report(df, text_col, name):
    n = len(df)
    n_unique = df[text_col].nunique()
    print(f"[{name}] {n} rows, {n_unique} unique texts, {n - n_unique} duplicated rows")
    if n != n_unique:
        dupes = (
            df[df.duplicated(subset=[text_col], keep=False)]
            .groupby(text_col)
            .size()
            .sort_values(ascending=False)
            .head(5)
        )
        print(f"  Top repeated texts ({name}):")
        for txt, cnt in dupes.items():
            print(f"    {cnt}x  {str(txt)[:80]}")

# --- Within-file duplicates ------------------------------------------------
_dup_report(dominant_topic_df, "Text", "TRAIN")
_dup_report(test_df,           "Text", "TEST")

# --- Same text, different labels? -----------------------------------------
for label_name, df in [("TRAIN", dominant_topic_df), ("TEST", test_df)]:
    same_text_diff_labels = (
        df.groupby("Text")[TOPIC_COLS]
          .nunique()
          .sum(axis=1)
          .gt(NUM_LABELS)      # > NUM_LABELS means at least one topic disagreed
    )
    n_conflict = int(same_text_diff_labels.sum())
    print(f"[{label_name}] duplicated texts with CONFLICTING labels: {n_conflict}")

# --- Train / test leakage --------------------------------------------------
train_texts = set(dominant_topic_df["Text"].astype(str))
test_texts  = set(test_df["Text"].astype(str))
overlap     = train_texts & test_texts
print(f"\n[TRAIN and TEST] texts appearing in BOTH files: {len(overlap)}")
if overlap:
    print("  These will be REMOVED from the training file so they do not inflate the test metric.")

# --- Deduplicate the training file BEFORE splitting -----------------------
before = len(dominant_topic_df)
# 1) drop any texts that also appear in the test corpus
dominant_topic_df = dominant_topic_df[~dominant_topic_df["Text"].isin(test_texts)]
# 2) drop exact-text duplicates within training (keep the first)
dominant_topic_df = dominant_topic_df.drop_duplicates(subset=["Text"]).reset_index(drop=True)
after = len(dominant_topic_df)
print(f"\nTraining file cleaned: {before} -> {after} rows ({before - after} removed).")

# 5. Train / Validation Split

With `USE_STRATIFIED_SPLIT = True` (recommended), we use `MultilabelStratifiedShuffleSplit` from the [`iterative-stratification`](https://github.com/trent-b/iterative-stratification) package. Standard `sklearn.train_test_split(stratify=y)` only supports single-label targets and would silently break on multi-label. Iterative stratification aims to balance every label's positive rate across the two splits, which matters when some topics are rare.

With `USE_STRATIFIED_SPLIT = False` we fall back to plain random splitting for reproducibility of the original notebook.

In [ ]:
X_all = dominant_topic_df["Text"].values
y_all = dominant_topic_df[TOPIC_COLS].values.astype(int)

if USE_STRATIFIED_SPLIT:
    splitter = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
    (train_idx, val_idx), = splitter.split(X_all, y_all)
else:
    train_idx, val_idx = train_test_split(
        np.arange(len(X_all)), test_size=0.2, random_state=SEED, shuffle=True
    )

train_df = dominant_topic_df.iloc[train_idx].reset_index(drop=True)
val_df   = dominant_topic_df.iloc[val_idx].reset_index(drop=True)

print(f"Split mode: {'multi-label stratified' if USE_STRATIFIED_SPLIT else 'random'}")
print(f"Train: {len(train_df)} rows | Val: {len(val_df)} rows")

per_class = pd.DataFrame({
    "train_pos": train_df[TOPIC_COLS].sum().astype(int),
    "val_pos":   val_df[TOPIC_COLS].sum().astype(int),
})
per_class["train_pos_rate"] = (per_class["train_pos"] / len(train_df)).round(3)
per_class["val_pos_rate"]   = (per_class["val_pos"]   / len(val_df)).round(3)
print("\nPer-topic positive counts and rates:")
print(per_class)

# Final leakage sanity check on the split itself
split_leak = set(train_df["Text"]).intersection(set(val_df["Text"]))
assert not split_leak, f"LEAK: {len(split_leak)} texts appear in both train and val"
print("\nNo train/val text overlap. Good.")

# 6. Tokenization

We wrap tokenisation in a small helper so we can rebuild the datasets identically for grid search, CV folds, and the final retraining. The label column is a length-6 float vector (multi-hot) — `BCEWithLogitsLoss` requires floats, not ints.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def make_tokenized_dataset(texts, labels, pks=None):
    """Build a torch-formatted HF Dataset ready for the DataLoader."""
    frame = {"Text": list(texts), "labels": [list(map(float, row)) for row in labels]}
    if pks is not None:
        frame["pk"] = list(pks)
    ds = Dataset.from_pandas(pd.DataFrame(frame))
    ds = ds.map(
        lambda ex: tokenizer(
            ex["Text"], padding="max_length", truncation=True, max_length=MAX_LENGTH
        ),
        batched=True,
    )
    ds = ds.remove_columns(["Text"])
    ds.set_format("torch")
    return ds

tokenized_train = make_tokenized_dataset(train_df["Text"].values, train_df[TOPIC_COLS].values)
tokenized_val   = make_tokenized_dataset(val_df["Text"].values,   val_df[TOPIC_COLS].values)
tokenized_test  = make_tokenized_dataset(
    test_df["Text"].values, test_df[TOPIC_COLS].values, pks=test_df["pk"].values
)

print(f"train: {len(tokenized_train)} | val: {len(tokenized_val)} | test: {len(tokenized_test)}")
print("Sample train row:")
print(f"  input_ids shape : {tokenized_train[0]['input_ids'].shape}")
print(f"  labels          : {tokenized_train[0]['labels']}")

# 7. Training Helper

One reusable function: build a fresh model, train with `BCEWithLogitsLoss` + early stopping, return the sigmoid probabilities on the validation set at the *best* epoch (lowest val loss). Used by:

* §8 (LR grid search) — called once per LR with the train/val split.
* §9 (threshold CV) — called once per fold.
* §10 (final retrain) — called once on the full training set, this time with `save_path` so we get a persisted checkpoint.

Fixed defaults (all set here, not tuned):
* `batch_size = 4` — modest for a 512-token sequence on Colab GPUs.
* `weight_decay = 0.01`, `warmup_ratio = 0.10`, `dropout = 0.1` — standard BERT fine-tuning values from the original paper.
* Gradient clipping at `max_norm = 1.0`.
* Loss is class-agnostic BCE (each topic independently) — no class weights by default, but the function accepts a `pos_weight` tensor if you want per-class up-weighting later.

In [ ]:
BATCH_SIZE            = 4
WEIGHT_DECAY          = 0.01
WARMUP_RATIO          = 0.10
DROPOUT               = 0.1
GRAD_CLIP             = 1.0
DEFAULT_NUM_EPOCHS    = 15
DEFAULT_ES_PATIENCE   = 3

def build_model():
    """Fresh multi-label classifier head on TwHIN-BERT."""
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_CHECKPOINT,
        num_labels=NUM_LABELS,
        problem_type="multi_label_classification",
        hidden_dropout_prob=DROPOUT,
        attention_probs_dropout_prob=DROPOUT,
    )
    return model.to(device)


@torch.no_grad()
def predict_probs(model, dataloader):
    """Return (probs, labels) as float32 numpy arrays, in dataloader order."""
    model.eval()
    all_probs, all_labels = [], []
    for batch in dataloader:
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        logits = model(input_ids, attention_mask=attention_mask).logits
        all_probs.append(torch.sigmoid(logits).cpu().numpy())
        all_labels.append(batch["labels"].numpy())
    return np.vstack(all_probs), np.vstack(all_labels)


def train_and_evaluate(
    train_ds,
    val_ds,
    lr,
    num_epochs=DEFAULT_NUM_EPOCHS,
    early_stopping_patience=DEFAULT_ES_PATIENCE,
    save_path=None,
    pos_weight=None,
    verbose=True,
):
    """One training run. Returns best val loss, val probabilities at the best epoch, and loss curves."""
    set_seed(SEED)

    model = build_model()
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)

    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY, eps=1e-8)
    num_steps = len(train_loader) * num_epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(WARMUP_RATIO * num_steps),
        num_training_steps=num_steps,
    )
    loss_fct = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    best_val_loss = float("inf")
    best_epoch    = -1
    best_val_probs = None
    best_val_labels = None
    epochs_since_improvement = 0
    train_losses, val_losses = [], []

    for epoch in range(num_epochs):
        # --- train ---
        model.train()
        epoch_train_losses = []
        for batch in train_loader:
            optimizer.zero_grad()
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["labels"].to(device).float()
            logits = model(input_ids, attention_mask=attention_mask).logits
            loss = loss_fct(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP)
            optimizer.step()
            scheduler.step()
            epoch_train_losses.append(loss.item())
        train_losses.append(float(np.mean(epoch_train_losses)))

        # --- val ---
        model.eval()
        val_loss_sum, n_val_batches = 0.0, 0
        with torch.no_grad():
            for batch in val_loader:
                input_ids      = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels         = batch["labels"].to(device).float()
                logits = model(input_ids, attention_mask=attention_mask).logits
                val_loss_sum += loss_fct(logits, labels).item()
                n_val_batches += 1
        avg_val_loss = val_loss_sum / max(n_val_batches, 1)
        val_losses.append(avg_val_loss)

        if verbose:
            print(f"epoch {epoch:>2} | train_loss={train_losses[-1]:.4f} | val_loss={avg_val_loss:.4f}")

        if avg_val_loss < best_val_loss:
            best_val_loss  = avg_val_loss
            best_epoch     = epoch
            best_val_probs, best_val_labels = predict_probs(model, val_loader)
            epochs_since_improvement = 0
            if save_path is not None:
                torch.save({
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "val_loss": best_val_loss,
                }, save_path)
        else:
            epochs_since_improvement += 1
            if epochs_since_improvement >= early_stopping_patience:
                if verbose:
                    print(f"Early stopping at epoch {epoch}")
                break

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "best_val_loss":  best_val_loss,
        "best_epoch":     best_epoch,
        "val_probs":      best_val_probs,
        "val_labels":     best_val_labels,
        "train_losses":   train_losses,
        "val_losses":     val_losses,
    }

# 8. Hyperparameter Search — Learning Rate Grid

Explicit and simple. We sweep the learning rate over the 4 values that dominate the BERT fine-tuning literature (`1e-5`, `2e-5`, `3e-5`, `5e-5`) and score each with **macro-F1 on the validation set at threshold 0.5**.

We deliberately don't tune the threshold here — that comes next (§9) via CV. Threshold selection at this stage would just reward whichever LR happens to overshoot on val at some specific threshold.

Everything else (batch size, dropout, weight decay, warmup, epochs) is fixed at the sensible defaults from §7. Total: **4 training runs**.

In [ ]:
LR_GRID = [1e-5, 2e-5, 3e-5, 5e-5]

grid_records = []
for lr in LR_GRID:
    print(f"\n=== lr = {lr:g} ===")
    result = train_and_evaluate(
        tokenized_train, tokenized_val,
        lr=lr,
        num_epochs=DEFAULT_NUM_EPOCHS,
        early_stopping_patience=DEFAULT_ES_PATIENCE,
        save_path=None,
        verbose=False,
    )
    binary_preds = (result["val_probs"] >= 0.5).astype(int)
    macro_f1     = f1_score(result["val_labels"], binary_preds, average="macro", zero_division=0)
    micro_f1     = f1_score(result["val_labels"], binary_preds, average="micro", zero_division=0)
    grid_records.append({
        "lr": lr,
        "best_epoch":    result["best_epoch"],
        "best_val_loss": result["best_val_loss"],
        "val_macro_f1@0.5": macro_f1,
        "val_micro_f1@0.5": micro_f1,
    })
    print(f"  best_epoch={result['best_epoch']} val_loss={result['best_val_loss']:.4f} "
          f"macro_f1@0.5={macro_f1:.3f} micro_f1@0.5={micro_f1:.3f}")

grid_df = pd.DataFrame(grid_records).sort_values("val_macro_f1@0.5", ascending=False)
print("\n=== Grid results (sorted by macro-F1) ===")
print(grid_df.to_string(index=False))

BEST_LR = float(grid_df.iloc[0]["lr"])
print(f"\nBest LR = {BEST_LR:g}")

# 9. Threshold Selection via K-Fold Cross-Validation

Why not pick the threshold on the validation set? Two reasons:

1. **Small validation set.** After the split it's ~50–60 rows. Sweeping the threshold on that would overfit the sweep to noise.
2. **Threshold and hyperparameters would double-dip on the same val split.** LR was already chosen using val; picking threshold there too compounds the optimistic bias.

Instead we run **5-fold CV on the training set** (the val set stays untouched here). In each fold we train a fresh model on 4/5 of `train_df` and predict probabilities on the held-out 1/5. Concatenating the held-out predictions gives us **out-of-fold (OOF) probabilities** for every training example — data the model never saw at prediction time. We sweep the threshold on those OOF probabilities.

**Compute cost:** 5 full training runs. Skip / reduce `n_splits` if this is too slow on Colab.

We report the best **global** threshold (one for all 6 topics) and, for reference, the best **per-topic** threshold. Per-topic is usually higher-F1 but overfits more on a small dataset; global is the safer default.

In [ ]:
N_CV_FOLDS = 5

X_train_texts = train_df["Text"].values
y_train_multi = train_df[TOPIC_COLS].values.astype(int)

if USE_STRATIFIED_SPLIT:
    cv = MultilabelStratifiedKFold(n_splits=N_CV_FOLDS, shuffle=True, random_state=SEED)
else:
    cv = KFold(n_splits=N_CV_FOLDS, shuffle=True, random_state=SEED)

oof_probs = np.zeros_like(y_train_multi, dtype=np.float32)
for fold, (tr_idx, ho_idx) in enumerate(cv.split(X_train_texts, y_train_multi)):
    print(f"\n=== fold {fold + 1}/{N_CV_FOLDS} ===")
    tr_ds = make_tokenized_dataset(X_train_texts[tr_idx], y_train_multi[tr_idx])
    ho_ds = make_tokenized_dataset(X_train_texts[ho_idx], y_train_multi[ho_idx])
    result = train_and_evaluate(
        tr_ds, ho_ds,
        lr=BEST_LR,
        num_epochs=DEFAULT_NUM_EPOCHS,
        early_stopping_patience=DEFAULT_ES_PATIENCE,
        save_path=None,
        verbose=False,
    )
    oof_probs[ho_idx] = result["val_probs"]
    print(f"  best_epoch={result['best_epoch']} best_val_loss={result['best_val_loss']:.4f}")

# --- Sweep GLOBAL threshold on OOF preds -----------------------------------
thresholds = np.round(np.arange(0.05, 0.95 + 1e-9, 0.05), 3)
global_scores = []
for t in thresholds:
    preds = (oof_probs >= t).astype(int)
    global_scores.append({
        "threshold":  float(t),
        "macro_f1":   f1_score(y_train_multi, preds, average="macro", zero_division=0),
        "micro_f1":   f1_score(y_train_multi, preds, average="micro", zero_division=0),
    })
global_df = pd.DataFrame(global_scores)
best_row  = global_df.loc[global_df["macro_f1"].idxmax()]
BEST_THRESHOLD_GLOBAL = float(best_row["threshold"])
print(f"\nBest GLOBAL threshold = {BEST_THRESHOLD_GLOBAL} "
      f"(OOF macro-F1 = {best_row['macro_f1']:.3f}, micro-F1 = {best_row['micro_f1']:.3f})")

# --- Also compute PER-TOPIC thresholds for reference ----------------------
per_topic_thresholds = []
for k in range(NUM_LABELS):
    best_t, best_f1 = 0.5, -1.0
    for t in thresholds:
        f1 = f1_score(y_train_multi[:, k], (oof_probs[:, k] >= t).astype(int), zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, float(t)
    per_topic_thresholds.append((best_t, best_f1))

BEST_THRESHOLDS_PER_TOPIC = np.array([t for t, _ in per_topic_thresholds], dtype=np.float32)
print("\nPer-topic best thresholds (from OOF):")
for i, (t, f1) in enumerate(per_topic_thresholds):
    print(f"  Topic_{i}: threshold={t}  OOF F1={f1:.3f}")

# --- Plot macro-F1 vs threshold -------------------------------------------
plt.style.use("fivethirtyeight")
plt.figure(figsize=(9, 4))
plt.plot(global_df["threshold"], global_df["macro_f1"], label="macro-F1")
plt.plot(global_df["threshold"], global_df["micro_f1"], label="micro-F1")
plt.axvline(BEST_THRESHOLD_GLOBAL, color="grey", linestyle="--",
            label=f"best global = {BEST_THRESHOLD_GLOBAL}")
plt.xlabel("Sigmoid threshold")
plt.ylabel("F1 on OOF predictions")
plt.title("OOF F1 vs threshold (5-fold CV on train)")
plt.legend()
plt.tight_layout()
plt.show()

# 10. Retrain Final Model

Now that we've chosen `BEST_LR` (from §8) and `BEST_THRESHOLD_GLOBAL` (from §9), retrain **once** on the full training set. This is the model we save to disk and use for downstream test inference.

In [ ]:
final_result = train_and_evaluate(
    tokenized_train, tokenized_val,
    lr=BEST_LR,
    num_epochs=DEFAULT_NUM_EPOCHS,
    early_stopping_patience=DEFAULT_ES_PATIENCE,
    save_path=BEST_MODEL_PATH,
    verbose=True,
)

print("\nBest epoch    :", final_result["best_epoch"])
print("Best val loss :", final_result["best_val_loss"])
print("Saved to      :", BEST_MODEL_PATH)

In [ ]:
# Loss curves
epochs = list(range(len(final_result["train_losses"])))
plt.figure(figsize=(9, 4))
plt.plot(epochs, final_result["train_losses"], label="train loss")
plt.plot(epochs, final_result["val_losses"],   label="val loss")
plt.xlabel("epoch")
plt.ylabel("BCE loss")
plt.title("Training curves")
plt.legend()
plt.tight_layout()
plt.show()

# 11. Load Best Checkpoint and Evaluate on the Validation Set

Sanity check that the saved checkpoint loads cleanly, then compute the val metrics using **the CV-selected threshold** from §9.

In [ ]:
checkpoint = torch.load(BEST_MODEL_PATH, map_location=device)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification",
).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print(f"Loaded checkpoint from epoch {checkpoint['epoch']} (val_loss={checkpoint['val_loss']:.4f})")

val_loader = DataLoader(tokenized_val, batch_size=BATCH_SIZE, shuffle=False)
val_probs, val_labels = predict_probs(model, val_loader)
val_preds = (val_probs >= BEST_THRESHOLD_GLOBAL).astype(int)

print(f"\n--- Val metrics @ threshold {BEST_THRESHOLD_GLOBAL} ---")
print(f"Accuracy      : {accuracy_score(val_labels, val_preds):.4f}")
print(f"F1 (macro)    : {f1_score(val_labels, val_preds, average='macro',    zero_division=0):.4f}")
print(f"F1 (micro)    : {f1_score(val_labels, val_preds, average='micro',    zero_division=0):.4f}")
print(f"F1 (weighted) : {f1_score(val_labels, val_preds, average='weighted', zero_division=0):.4f}")
print("\nPer-topic:")
for k in range(NUM_LABELS):
    print(f"  Topic_{k}: acc={accuracy_score(val_labels[:, k], val_preds[:, k]):.3f}  "
          f"F1={f1_score(val_labels[:, k], val_preds[:, k], zero_division=0):.3f}  "
          f"support={int(val_labels[:, k].sum())}")

# 12. Predict on the Held-out Test Set

Apply the same model + threshold to the unseen test corpus. Because we removed any test-file texts from the training file back in §4, this is a genuine held-out evaluation (barring within-test duplicates, which are just repeats of unseen examples).

In [ ]:
test_loader = DataLoader(tokenized_test, batch_size=BATCH_SIZE, shuffle=False)

model.eval()
test_probs_batches, test_pks_batches = [], []
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Predicting on test"):
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        logits = model(input_ids, attention_mask=attention_mask).logits
        test_probs_batches.append(torch.sigmoid(logits).cpu().numpy())
        test_pks_batches.append(batch["pk"].cpu().numpy())

test_probs = np.vstack(test_probs_batches)
test_pks   = np.concatenate(test_pks_batches)
test_labels = test_df.set_index("pk").loc[test_pks, TOPIC_COLS].values.astype(int)

# Predictions using the CV-selected global threshold.
test_preds = (test_probs >= BEST_THRESHOLD_GLOBAL).astype(int)

# Build a tidy prediction DataFrame keyed by pk.
prob_cols = [f"Topic_{i}_Prob" for i in range(NUM_LABELS)]
prediction_df = pd.DataFrame(test_probs, columns=prob_cols)
prediction_df["pk"] = test_pks
prediction_df["Text"] = prediction_df["pk"].map(test_df.set_index("pk")["Text"])
prediction_df = prediction_df[["pk", "Text"] + prob_cols]
display(prediction_df.head())

print(f"\n--- Test metrics @ threshold {BEST_THRESHOLD_GLOBAL} ---")
print(f"Accuracy      : {accuracy_score(test_labels, test_preds):.4f}")
print(f"F1 (macro)    : {f1_score(test_labels, test_preds, average='macro',    zero_division=0):.4f}")
print(f"F1 (micro)    : {f1_score(test_labels, test_preds, average='micro',    zero_division=0):.4f}")
print(f"F1 (weighted) : {f1_score(test_labels, test_preds, average='weighted', zero_division=0):.4f}")
print("\nPer-topic (global threshold):")
for k in range(NUM_LABELS):
    print(f"  Topic_{k}: acc={accuracy_score(test_labels[:, k], test_preds[:, k]):.3f}  "
          f"F1={f1_score(test_labels[:, k], test_preds[:, k], zero_division=0):.3f}  "
          f"support={int(test_labels[:, k].sum())}")

# Optional: also report using the per-topic thresholds selected in §9.
test_preds_pt = (test_probs >= BEST_THRESHOLDS_PER_TOPIC[None, :]).astype(int)
print(f"\n--- Test metrics with PER-TOPIC thresholds {BEST_THRESHOLDS_PER_TOPIC.tolist()} ---")
print(f"F1 (macro)    : {f1_score(test_labels, test_preds_pt, average='macro',    zero_division=0):.4f}")
print(f"F1 (micro)    : {f1_score(test_labels, test_preds_pt, average='micro',    zero_division=0):.4f}")

# 13. Prediction Outcome Scatter

Every point is one (document, topic) pair. Colour = confusion-matrix cell (TP / TN / FP / FN) at the chosen threshold. Useful for eyeballing where the model is under/over-confident.

In [ ]:
COLOR_TP = "#FF9D00"
COLOR_TN = "#00A2F3"
COLOR_FP = "#4B2362"
COLOR_FN = "#CE4763"

points = []
for i, prob_col in enumerate(prob_cols):
    probs = prediction_df[prob_col].values
    truths = test_df.set_index("pk").loc[prediction_df["pk"], f"Topic_{i}"].values.astype(int)
    preds = (probs >= BEST_THRESHOLD_GLOBAL).astype(int)
    for pk, p, t, pr in zip(prediction_df["pk"].values, probs, truths, preds):
        if   t == 1 and pr == 1: c = COLOR_TP
        elif t == 0 and pr == 0: c = COLOR_TN
        elif t == 0 and pr == 1: c = COLOR_FP
        else:                    c = COLOR_FN
        points.append((pk, p, c))

fig, ax = plt.subplots(figsize=(14, 6))
for pk, p, c in points:
    ax.scatter(pk, p, color=c, alpha=0.7, s=20)
ax.axhline(BEST_THRESHOLD_GLOBAL, color="grey", linestyle="dashed",
           linewidth=2, label=f"threshold = {BEST_THRESHOLD_GLOBAL}")
ax.set_title("Predicted probability per (document, topic)")
ax.set_xlabel("Document ID (pk)")
ax.set_ylabel("Predicted probability")
ax.set_ylim(0, 1)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(handles=[
    Line2D([0], [0], marker="o", color="w", label="True Positive",  markerfacecolor=COLOR_TP, markersize=8),
    Line2D([0], [0], marker="o", color="w", label="True Negative",  markerfacecolor=COLOR_TN, markersize=8),
    Line2D([0], [0], marker="o", color="w", label="False Positive", markerfacecolor=COLOR_FP, markersize=8),
    Line2D([0], [0], marker="o", color="w", label="False Negative", markerfacecolor=COLOR_FN, markersize=8),
    Line2D([0], [0], color="grey", linestyle="dashed", label=f"threshold = {BEST_THRESHOLD_GLOBAL}"),
], title="Outcome")
plt.tight_layout()
plt.show()